# Assignment 8 — Movie Recommendation Systems

**Juan Luna Santibanez**  
**Miami Dade College · CAI2100C-2267-6908 · Machine Learning Foundations**  
**Professor Beth Muturi**

## My assignment

I implement and compare **two recommendation approaches** in the two final tasks of my course notebook:

- **Task 1 — Content-Based Filtering (IMDb Extensive):** I represent genres, directors, actors, and descriptions with TF-IDF and recommend related films with cosine similarity.
- **Task 2 — Collaborative Filtering (IMDb Extensive):** I compare available demographic-cohort rating averages using Pearson correlation. These are *group-level estimates*, **not individual-user recommendations or Surprise SVD**.

I also include a **separately labeled classroom extension** using MovieLens/TMDB, because the instructor's follow-up email requested “both datasets” without naming them and the class ZIP contains MovieLens/TMDB tables. This extra section is a classroom comparison, not a third assignment or a claim that IMDb and MovieLens are the same dataset.

**My sources:** [IMDb Extensive on Kaggle](https://www.kaggle.com/datasets/stefanoleone992/imdb-extensive-dataset); [Rounak Banik's public classroom-method reference](https://www.kaggle.com/code/rounakbanik/movie-recommender-systems); instructor-provided `Assignment 8-machinelearning.zip`. My notebook does not redistribute either dataset.

**Project links:** [Google Colab](https://colab.research.google.com/github/SupaHumano/CAI2100C-Movie-Recommendation-Systems/blob/main/notebooks/Assignment_8_Movie_Recommendation_Systems.ipynb) · [Jupyter on GitHub](https://github.com/SupaHumano/CAI2100C-Movie-Recommendation-Systems/blob/main/notebooks/Assignment_8_Movie_Recommendation_Systems.ipynb) · [IMDb source](https://www.kaggle.com/datasets/stefanoleone992/imdb-extensive-dataset) · [Interactive teaching demo](https://supahumano.github.io/CAI2100C-Movie-Recommendation-Systems/) · [Oral presentation guide](https://github.com/SupaHumano/CAI2100C-Movie-Recommendation-Systems/blob/main/docs/PRESENTATION_GUIDE.md).

**Scientific integrity:** I distinguish authentic IMDb results from MovieLens classroom results, and I never use the fictional titles on my educational website as measured evidence. Output cells must be regenerated with authentic source CSVs before I submit the executed notebook.


## Running my notebook in Colab, Kaggle or Jupyter

1. I open this notebook through Colab or upload the downloaded `.ipynb` to Colab/Jupyter/Kaggle Code.
2. I obtain **`IMDb movies.csv`** and **`IMDb ratings.csv`** from the Kaggle dataset linked above and upload them beside the notebook. In Colab I can use the Files sidebar (`/content`). In Kaggle Code I attach the dataset to my session. I can also place them in `data/` locally.
3. For my **optional classroom extension**, I upload the unchanged instructor ZIP named **`Assignment 8-machinelearning.zip`**, or extract its four CSV files into `data/`. These tables are not interchangeable with the IMDb CSVs.
4. I choose **Run all** and check that the primary IMDb recommendations, group-rating validation, MovieLens classroom results and final summary execute without errors. I save the executed `.ipynb` and submit that version to Canvas.

**Why two sources?** The *two final assignment tasks* each reference **IMDb Extensive**; the earlier lesson demonstrates **MovieLens/TMDB**. My classroom extension documents both sources because of the email. This is **one notebook for one assignment**.

**Important:** If I am missing the classroom files, the extension raises an explicit error. If I am missing the IMDb files, the main assignment raises an explicit error. I do not invent movies or scores in place of missing academic datasets.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os, sys, platform, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from IPython.display import display

np.random.seed(42)
ROOTS = [Path.cwd(), Path('/content'), Path.cwd() / 'data', Path('/content/data')]
ROOTS = list(dict.fromkeys(p for p in ROOTS if p.is_dir()))
print('Python:', platform.python_version(), '| pandas:', pd.__version__)
print('Working data locations:', ', '.join(str(p) for p in ROOTS))

# Shared file discovery for both independent datasets.
def find_file(filename):
    for root in ROOTS:
        exact=root/filename
        if exact.is_file():return exact
    for root in ROOTS:
        matches=[p for p in root.glob('**/'+filename) if p.is_file()]
        if matches:return sorted(matches,key=lambda p:len(str(p)))[0]
    return None

# Shared TF-IDF cosine ranking; excludes the query title itself.
def recommend_by_content(frame,matrix,selected_title,k=10):
    found=np.flatnonzero(frame.title.astype(str).str.casefold().eq(selected_title.casefold()))
    if not len(found):raise ValueError('Unknown title: '+selected_title)
    target=int(found[0]);sims=linear_kernel(matrix[target],matrix).ravel()
    ids=np.argsort(-sims,kind='stable');ids=ids[ids!=target][:k]
    result=frame.iloc[ids][['title']].copy();result['cosine_similarity']=sims[ids]
    return result.reset_index(drop=True)


## Part I — My required IMDb Extensive tasks

I use the two assigned Kaggle files, `IMDb movies.csv` and `IMDb ratings.csv`. I first validate my inputs and document their exact SHA-256 checksums and row counts. I then select up to 3,000 high-vote titles using a reproducible rule. This is a **research subsample**, not a claim that IMDb contains only 3,000 films.

**Task 1:** compare movie content (TF-IDF and cosine similarity).  
**Task 2:** compare demographic average-rating patterns (cohort-level collaborative approximation with a held-out MAE/RMSE test and a training-only baseline).


In [ ]:
IMDB_MOVIES=find_file('IMDb movies.csv')
IMDB_RATINGS=find_file('IMDb ratings.csv')
if IMDB_MOVIES is None or IMDB_RATINGS is None:
    raise FileNotFoundError('Upload BOTH authentic IMDb movies.csv and IMDb ratings.csv from the assigned Kaggle IMDb Extensive dataset to the Colab Files panel, then Run all.')

imdb_movies_raw=pd.read_csv(IMDB_MOVIES,low_memory=False)
imdb_ratings_raw=pd.read_csv(IMDB_RATINGS,low_memory=False)
IMDB_FIELDS={'imdb_title_id','title','genre','description','director','actors','votes','avg_vote'}
COHORTS={
 'Women 18–29':('females_18age_avg_vote','females_18age_votes'),
 'Women 30–44':('females_30age_avg_vote','females_30age_votes'),
 'Women 45+':('females_45age_avg_vote','females_45age_votes'),
 'Men 18–29':('males_18age_avg_vote','males_18age_votes'),
 'Men 30–44':('males_30age_avg_vote','males_30age_votes'),
 'Men 45+':('males_45age_avg_vote','males_45age_votes')}
cohort_cols=[name for pair in COHORTS.values() for name in pair]
assert IMDB_FIELDS.issubset(imdb_movies_raw.columns), 'Missing IMDb movies columns: '+str(IMDB_FIELDS-set(imdb_movies_raw.columns))
assert {'imdb_title_id',*cohort_cols}.issubset(imdb_ratings_raw.columns), 'Missing columns in IMDb ratings.csv'
print('IMDb raw movie rows:',len(imdb_movies_raw),'| IMDb raw ratings rows:',len(imdb_ratings_raw))
print('Files:',IMDB_MOVIES.name,'and',IMDB_RATINGS.name)
print('SHA256 movies:',hashlib.sha256(IMDB_MOVIES.read_bytes()).hexdigest())
print('SHA256 ratings:',hashlib.sha256(IMDB_RATINGS.read_bytes()).hexdigest())

### B1. IMDb data cleaning and reproducible selection

Select a maximum of 3,000 titles with at least 1,000 overall votes, ordered by popularity (vote count). This bound avoids unnecessary memory use and makes the practical exercise reproducible. The **full raw row counts** are shown above; the filtered subset is **not** the entire dataset.

In [ ]:
imdb_movies=imdb_movies_raw.copy()
imdb_movies['votes']=pd.to_numeric(imdb_movies['votes'],errors='coerce').fillna(0)
imdb_movies['avg_vote']=pd.to_numeric(imdb_movies['avg_vote'],errors='coerce')
imdb_movies=imdb_movies[imdb_movies.votes.ge(1000)&imdb_movies.title.notna()&imdb_movies.imdb_title_id.notna()].copy()
imdb_movies=imdb_movies.sort_values(['votes','imdb_title_id'],ascending=[False,True]).drop_duplicates('imdb_title_id').head(3000)
for col in ['title','genre','description','director','actors']:
    imdb_movies[col]=imdb_movies[col].fillna('').astype(str)
# Ignore duplicated keys in the right-hand table, not the movie rows themselves.
imdb_ratings_unique=imdb_ratings_raw[['imdb_title_id']+cohort_cols].drop_duplicates('imdb_title_id')
imdb_movies=imdb_movies.merge(imdb_ratings_unique,on='imdb_title_id',how='left',validate='one_to_one').reset_index(drop=True)
assert len(imdb_movies)>=5
print('IMDb titles in reproducible analysis subset:',len(imdb_movies))
display(imdb_movies[['title','genre','votes','avg_vote']].head())

### B2. Task 1 — Content-Based (IMDb)

Movie text features = genre + director + actors + description. We compute TF-IDF (unigrams/bigrams) and cosine similarity only for the query title. This demonstrates the difference between **text resemblance** and **rating-based prediction**.

In [ ]:
imdb_text=(imdb_movies.genre.str.replace(',',' ',regex=False)+' '+
           imdb_movies.director.str.replace(',',' ',regex=False)+' '+
           imdb_movies.actors.str.replace(',',' ',regex=False)+' '+imdb_movies.description)
imdb_vectorizer=TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_features=6000)
imdb_features=imdb_vectorizer.fit_transform(imdb_text)
imdb_title=str(imdb_movies.iloc[0].title)
imdb_content=recommend_by_content(imdb_movies,imdb_features,imdb_title,k=10)
print('Selected IMDb title:',imdb_title)
display(imdb_content.round(4))
assert len(imdb_content)==10
fig,ax=plt.subplots(figsize=(9,4.5));table=imdb_content.iloc[::-1]
ax.barh(table.title.str.slice(0,38),table.cosine_similarity)
ax.set(title='IMDb Extensive — Content-Based recommendations',xlabel='TF-IDF cosine similarity')
fig.tight_layout();plt.show()

### B3. Task 2 — Collaborative approximation using IMDb rating cohorts

The 6×N matrix represents **group-average ratings** (1–10), withholding cells with fewer than 30 votes in a group. Pearson correlation between group rows is computed only where there are at least 15 co-rated titles. The model uses positive similarities to estimate missing group means.

**Evaluation:** hold out about 15% of valid group-average cells while leaving enough data in each group and movie. Fit similarities using *training cells only* and compare predictions with withheld observations. Also compare against a simple **training cohort mean** baseline. This is **not** a user-level recommender.

In [ ]:
def make_cohort_matrix(frame,min_votes=30):
    rows=[]
    for mean_col,votes_col in COHORTS.values():
        mean=pd.to_numeric(frame[mean_col],errors='coerce')
        nvotes=pd.to_numeric(frame[votes_col],errors='coerce').fillna(0)
        rows.append(mean.where(mean.between(1,10)&nvotes.ge(min_votes)).to_numpy(float))
    return pd.DataFrame(rows,index=list(COHORTS),columns=frame.imdb_title_id.tolist())

def group_similarity(groups,min_overlap=15):
    x=groups.to_numpy(float);n=x.shape[0];sim=np.eye(n)
    for i in range(n):
        for j in range(i+1,n):
            valid=np.isfinite(x[i])&np.isfinite(x[j])
            if valid.sum()>=min_overlap and np.std(x[i,valid])>1e-12 and np.std(x[j,valid])>1e-12:
                s=np.corrcoef(x[i,valid],x[j,valid])[0,1]
                sim[i,j]=sim[j,i]=float(np.clip(s,-1,1)) if np.isfinite(s) else 0.0
            else:sim[i,j]=sim[j,i]=0.0
    return pd.DataFrame(sim,index=groups.index,columns=groups.index)

def predict_group_average(groups,similarity,row,col):
    x=groups.to_numpy(float)
    group_av=np.nanmean(x,axis=1)
    global_mean=float(np.nanmean(x))
    base=float(group_av[row]) if np.isfinite(group_av[row]) else global_mean
    w=similarity.iloc[row].to_numpy(float).copy();w[row]=0
    good=(w>0)&np.isfinite(x[:,col])&np.isfinite(group_av)
    if not good.any():return float(np.clip(base,1,10))
    result=base+np.dot(w[good],x[good,col]-group_av[good])/w[good].sum()
    return float(np.clip(result,1,10))

def evaluate_groups(groups,seed=42,fraction=0.15):
    x=groups.to_numpy(float)
    pairs=np.argwhere(np.isfinite(x))
    if len(pairs)<60:raise ValueError('Need at least 60 observed demographic averages for validation')
    rng=np.random.default_rng(seed);rng.shuffle(pairs)
    train=groups.copy()
    withheld=[];max_holdout=max(1,round(len(pairs)*fraction))
    for row,col in pairs:
        if len(withheld)>=max_holdout:break
        if train.iloc[row].notna().sum()<=15 or train.iloc[:,col].notna().sum()<=2:continue
        withheld.append((int(row),int(col),float(groups.iloc[row,col])))
        train.iloc[row,col]=np.nan
    if not withheld:raise ValueError('Unable to select any evaluable holdout')
    sims=group_similarity(train)
    y_true=np.array([value for _,_,value in withheld])
    y_pred=np.array([predict_group_average(train,sims,row,col) for row,col,_ in withheld])
    group_means=np.nanmean(train.to_numpy(float),axis=1)
    baseline=np.array([group_means[row] for row,_,_ in withheld])
    diff=y_pred-y_true
    return {'Held-out group means':len(y_true),'MAE':float(np.mean(abs(diff))),
            'RMSE':float(np.sqrt(np.mean(diff**2))),
            'Training-group-mean baseline MAE':float(np.mean(abs(baseline-y_true)))},train

imdb_groups=make_cohort_matrix(imdb_movies)
imdb_sim=group_similarity(imdb_groups)
print('IMDb matrix dimensions (groups × movies):',imdb_groups.shape)
display(imdb_sim.round(3))
imdb_scores,imdb_train_groups=evaluate_groups(imdb_groups)
display(pd.DataFrame([imdb_scores]).T.rename(columns={0:'Value'}))
assert np.isfinite(imdb_scores['MAE'])
fig,ax=plt.subplots(figsize=(7,5))
heat=ax.imshow(imdb_sim.to_numpy(),vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(6),imdb_sim.columns,rotation=45,ha='right')
ax.set_yticks(range(6),imdb_sim.index)
fig.colorbar(heat,ax=ax,label='Pearson correlation');fig.tight_layout();plt.show()

In [ ]:
# Example estimates for a named cohort. If no cells are missing, demonstrate estimates for sufficiently observed movies.
chosen_group='Women 18–29'; row=list(imdb_groups.index).index(chosen_group)
available=np.flatnonzero(imdb_groups.iloc[row].isna().to_numpy() & (imdb_groups.notna().sum(axis=0).to_numpy()>=2))
reason='Missing group-average estimates'
if not len(available):
    available=np.flatnonzero(imdb_groups.notna().sum(axis=0).to_numpy()>=3)
    reason='Illustrative group-average estimates for observed movie IDs (not unobserved personal preferences)'
if not len(available):raise ValueError('Not enough IMDb group-mean observations for ranking')
est=np.array([predict_group_average(imdb_groups,imdb_sim,row,int(j)) for j in available])
order=np.argsort(-est)[:10]
keys=imdb_groups.columns[available[order]]
imdb_group_top=imdb_movies.set_index('imdb_title_id').loc[keys,['title','genre','avg_vote']].copy()
imdb_group_top['estimated_group_average']=est[order]
print('IMDb group:',chosen_group,'|',reason)
display(imdb_group_top.reset_index())

## Part II — Classroom extension: MovieLens/TMDB

**Why I included this extension.** My teacher's classroom example uses *The Movies Dataset* (MovieLens ratings and TMDB metadata); her final two tasks use IMDb Extensive. After her email said “Please use both datasets,” I chose to preserve the classroom source and make its separate results visible below. I do not conflate these sources or their rating scales.

I inspect all four provided tables and read only the **first 20,000 rows** from the large `ratings.csv` to avoid loading a potentially huge file into memory. My predictive evaluation uses `ratings_small.csv` and never treats the truncated full-ratings file as a complete dataset.

This is an **additional comparison**, not a replacement for Task 1 or Task 2. I use user-neighborhood collaborative filtering here; the lecture's original Surprise/SVD example is another legitimate implementation and **is not the model I claim to have reproduced**.


In [ ]:
CLASSROOM_FILES = ('movies_metadata.csv', 'links_small.csv', 'ratings_small.csv', 'ratings.csv')

class_zip = find_file('Assignment 8-machinelearning.zip')

def read_classroom(filename, nrows=None):
    if class_zip is not None:
        with ZipFile(class_zip) as archive:
            matches=[n for n in archive.namelist() if Path(n).name == filename and '__MACOSX' not in n and not n.startswith('._')]
            if len(matches) != 1:
                raise FileNotFoundError(f'Expected one {filename} in {class_zip}; found {len(matches)}')
            with archive.open(matches[0]) as stream:
                return pd.read_csv(stream, low_memory=False, nrows=nrows)
    csv_path = find_file(filename)
    if csv_path is None:
        raise FileNotFoundError('Classroom dataset missing: upload Assignment 8-machinelearning.zip or extracted ' + filename)
    return pd.read_csv(csv_path, low_memory=False, nrows=nrows)

ml_metadata = read_classroom('movies_metadata.csv')
ml_links = read_classroom('links_small.csv')
ml_ratings = read_classroom('ratings_small.csv')
ml_large_sample = read_classroom('ratings.csv', nrows=20000)
assert {'id','title','overview'}.issubset(ml_metadata.columns)
assert {'movieId','tmdbId'}.issubset(ml_links.columns)
assert {'userId','movieId','rating'}.issubset(ml_ratings.columns)
assert {'userId','movieId','rating'}.issubset(ml_large_sample.columns)
assert ml_ratings['rating'].between(0.5,5).all()
ml_summary = pd.DataFrame([
 ('movies_metadata.csv',len(ml_metadata),'complete file'),
 ('links_small.csv',len(ml_links),'complete file'),
 ('ratings_small.csv',len(ml_ratings),'complete file'),
 ('ratings.csv',len(ml_large_sample),'first 20,000 rows ONLY (sample)')
],columns=['Classroom file','Rows read','Coverage'])
display(ml_summary)
print('Users in ratings_small:',ml_ratings.userId.nunique())
print('Rated MovieLens IDs:',ml_ratings.movieId.nunique())
print('Data source:', str(class_zip) if class_zip else 'Extracted classroom CSVs')

### A2. Content-Based Filtering (classroom example)

The instructor's example uses a movie overview (and where available, its tagline). We keep that idea and use `links_small.csv` to select the MovieLens/TMDB classroom subset. **TF-IDF** makes the text numeric; **cosine similarity** measures similarity between the chosen title and candidate films. We calculate similarities **on demand**, not a large dense all-to-all matrix.

In [ ]:
ml_metadata = ml_metadata.copy()
ml_metadata['tmdb_numeric'] = pd.to_numeric(ml_metadata['id'], errors='coerce')
ml_links = ml_links.copy()
ml_links['tmdb_numeric'] = pd.to_numeric(ml_links['tmdbId'], errors='coerce')
subset_ids = set(ml_links['tmdb_numeric'].dropna().astype('int64').tolist())
ml_small = ml_metadata.loc[ml_metadata['tmdb_numeric'].isin(subset_ids)].copy()
ml_small = ml_small.dropna(subset=['title']).drop_duplicates('tmdb_numeric').reset_index(drop=True)
ml_small['overview'] = ml_small['overview'].fillna('').astype(str)
ml_small['tagline'] = ml_small.get('tagline',pd.Series('',index=ml_small.index)).fillna('').astype(str)
ml_small['description'] = (ml_small['overview']+' '+ml_small['tagline']).str.strip()
ml_small = ml_small.loc[ml_small['description'].str.len()>10].reset_index(drop=True)
assert len(ml_small) >= 5, 'Not enough classroom movies with useful descriptions'
ml_tfidf = TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_features=6500)
ml_X = ml_tfidf.fit_transform(ml_small['description'])

ml_title = 'The Dark Knight' if ml_small['title'].eq('The Dark Knight').any() else str(ml_small.iloc[0]['title'])
ml_content = recommend_by_content(ml_small,ml_X,ml_title,k=10)
print('Classroom selected movie:',ml_title,'| searchable movies:',len(ml_small))
display(ml_content.round(4))
assert len(ml_content) == 10

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
chart=ml_content.iloc[::-1]
ax.barh(chart['title'].str.slice(0,38),chart['cosine_similarity'])
ax.set(xlabel='TF-IDF cosine similarity',title='Classroom MovieLens/TMDB — Content-Based recommendations')
fig.tight_layout();plt.show()

### A3. Collaborative Filtering (user-based, MovieLens)

`ratings_small.csv` stores **individual** ratings. We split events 80/20 (`random_state=42`), calculate **user-user** similarities from **training data only**, and predict held-out ratings with up to 25 positively similar neighbors. This is neighborhood-based collaborative filtering, **not Surprise SVD/matrix factorization**. The course notebook also illustrates Surprise/SVD; here we use a compatible installation-free technique for a reliable and explainable single-notebook submission.

In [ ]:
ml_events = ml_ratings[['userId','movieId','rating']].dropna().copy()
ml_events = ml_events.groupby(['userId','movieId'],as_index=False)['rating'].mean()
ml_train,ml_test = train_test_split(ml_events,test_size=0.20,random_state=42)
ml_users = np.sort(ml_train.userId.unique()); ml_items = np.sort(ml_train.movieId.unique())
ml_uid={u:i for i,u in enumerate(ml_users)}; ml_mid={m:i for i,m in enumerate(ml_items)}
ml_rr=ml_train.userId.map(ml_uid).to_numpy(); ml_cc=ml_train.movieId.map(ml_mid).to_numpy()
ml_vv=ml_train.rating.to_numpy(dtype=float)
ml_user_av=ml_train.groupby('userId').rating.mean();ml_global=float(ml_train.rating.mean())
ml_means = np.array([ml_user_av.get(u,ml_global) for u in ml_users])
ml_mat=csr_matrix((ml_vv,(ml_rr,ml_cc)),shape=(len(ml_users),len(ml_items)))
ml_centered=csr_matrix((ml_vv-ml_means[ml_rr],(ml_rr,ml_cc)),shape=ml_mat.shape)
ml_sim=cosine_similarity(ml_centered);np.fill_diagonal(ml_sim,0)
ml_columns=ml_mat.tocsc()

def predict_rating(user,movie):
    if user not in ml_uid: return ml_global
    row=ml_uid[user]
    if movie not in ml_mid: return float(np.clip(ml_means[row],0.5,5))
    col=ml_mid[movie]
    ratings_col=ml_columns.getcol(col)
    neighbors=ratings_col.indices
    ratings_values=ratings_col.data
    pos=ml_sim[row,neighbors] > 0
    neighbors=neighbors[pos];ratings_values=ratings_values[pos]
    if not len(neighbors):return float(np.clip(ml_means[row],0.5,5))
    top=np.argsort(-ml_sim[row,neighbors])[:25]
    neighbors=neighbors[top];ratings_values=ratings_values[top]
    w=ml_sim[row,neighbors]
    est=ml_means[row]+np.dot(w,ratings_values-ml_means[neighbors])/w.sum()
    return float(np.clip(est,0.5,5))

# Only evaluate ratings with known training users and items. Report coverage.
ml_eval=ml_test.loc[ml_test.userId.isin(ml_uid)&ml_test.movieId.isin(ml_mid)].copy()
assert not ml_eval.empty
ml_predictions = np.array([predict_rating(u,m) for u,m in ml_eval[['userId','movieId']].itertuples(index=False,name=None)])
ml_truth=ml_eval.rating.to_numpy(float)
ml_errors=ml_predictions-ml_truth
ml_mae=float(np.mean(np.abs(ml_errors)))
ml_rmse=float(np.sqrt(np.mean(ml_errors**2)))
# Baseline is the user average from training data only; same evaluation rows.
ml_baseline=np.array([ml_means[ml_uid[u]] for u in ml_eval.userId])
ml_baseline_mae=float(np.mean(np.abs(ml_baseline-ml_truth)))
ml_metrics=pd.DataFrame([('Train ratings',len(ml_train)),('Held-out test ratings',len(ml_test)),
 ('Evaluable test ratings',len(ml_eval)),('Coverage',len(ml_eval)/len(ml_test)),
 ('User-based MAE',ml_mae),('User-based RMSE',ml_rmse),('User-mean baseline MAE',ml_baseline_mae)],columns=['Metric','Value'])
display(ml_metrics)
assert np.isfinite(ml_mae) and np.isfinite(ml_rmse)

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.5))
ax.hist(ml_errors,bins=25)
ax.axvline(0,linestyle='--',color='black')
ax.set(xlabel='Predicted rating minus actual rating',ylabel='Count',title='Classroom user-based collaborative — held-out errors')
fig.tight_layout();plt.show()

## Part C — Results summary, interpretation, and submission checks

The two sources use different rating scales and objectives. **Do not claim** IMDb MAE is better or worse than MovieLens MAE by comparing the raw numbers directly. The IMDb model predicts *group means* (1–10); the classroom model predicts *individual user ratings* (0.5–5). TF-IDF cosine similarities are not rating prediction errors.

The classroom notebook originally illustrated Surprise SVD; our user-based implementation is a different collaborative method. Both satisfy the conceptual distinction but should not be described as the same algorithm.


In [ ]:
final_comparison=pd.DataFrame([
 {'Source':'MovieLens/TMDB classroom','Approach':'Content-Based','Output':f'{len(ml_content)} related titles','Metric':'TF-IDF cosine similarity'},
 {'Source':'MovieLens/TMDB classroom','Approach':'User-Based Collaborative','Output':f'{len(ml_eval)} held-out ratings','Metric':f'MAE {ml_mae:.4f}; RMSE {ml_rmse:.4f}; baseline MAE {ml_baseline_mae:.4f}'},
 {'Source':'IMDb Extensive final task','Approach':'Content-Based','Output':f'{len(imdb_content)} related titles','Metric':'TF-IDF cosine similarity'},
 {'Source':'IMDb Extensive final task','Approach':'Cohort Collaborative approximation','Output':f'{imdb_scores["Held-out group means"]} held-out means','Metric':f'MAE {imdb_scores["MAE"]:.4f}; RMSE {imdb_scores["RMSE"]:.4f}; baseline MAE {imdb_scores["Training-group-mean baseline MAE"]:.4f}'}])
display(final_comparison)
print('PASS — Both data sources processed; two content-based and two rating-pattern methods produced outputs.')

## My interpretation and conclusions

I completed **two assignment methods** with **IMDb Extensive** and added the separately labeled **MovieLens/TMDB classroom extension** to address the request to consider both sources. Both datasets can support content-based recommendations, but their available ratings differ:

- In **IMDb Extensive**, I can compare rating patterns across **six demographic cohorts**, not predict a particular person's taste. The MAE/RMSE scores evaluate held-out **cohort means on the 1–10 scale**, not personal ratings.
- In **MovieLens/TMDB**, I work with individual `userId–movieId–rating` records and evaluate a **user-based nearest-neighbor model on the 0.5–5 scale**.
- In each method I hold out observations for evaluation and compare prediction error against a simple **training-only mean baseline**. I do not compare the raw MAE/RMSE across the two rating scales as if they measured identical targets.

My interactive website is a **separate fictional teaching demonstration**, not the source of my authentic numerical findings. The notebook outputs, plots and numerical metrics are the scientific evidence when I execute it using the authentic datasets.

**What I learned.** Content-based filtering starts with what a movie is about. Collaborative filtering starts with rating patterns. The choice of dataset determines whether I can model individual preferences or only group-level information. My approach documents this distinction instead of overstating personalization.

**References:** Beth Muturi, CAI2100C *Machine Learning Foundations*, Assignment 8 notebook and class ZIP; [Stefano Leone, IMDb Extensive](https://www.kaggle.com/datasets/stefanoleone992/imdb-extensive-dataset); [Rounak Banik, Movie Recommender Systems](https://www.kaggle.com/code/rounakbanik/movie-recommender-systems); [scikit-learn TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html); [scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity).
